# Tutorial: Principal Stratification & Dynamic Treatment Regimes

This tutorial demonstrates two advanced causal inference methods:

1. **Principal Stratification**: Handling noncompliance in RCTs (CACE/LATE estimation)
2. **Dynamic Treatment Regimes (DTR)**: Optimal sequential treatment decisions

**Key Concepts**:
- **CACE = LATE**: Complier Average Causal Effect equals Local Average Treatment Effect
- **Principal Strata**: Compliers, Always-takers, Never-takers
- **Backward Induction**: Q-learning from final stage to first
- **Double Robustness**: A-learning consistent if either model correct

---

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import sys
sys.path.insert(0, '../../src')

# Principal Stratification
from causal_inference.principal_stratification import (
    cace_2sls,
    wald_estimator,
    ps_bounds_monotonicity,
    ps_bounds_no_assumption,
)

# Dynamic Treatment Regimes
from causal_inference.dtr import (
    DTRData,
    q_learning_single_stage,
    a_learning_single_stage,
)

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

---

## Part 1: Principal Stratification Concepts

### The Problem: Noncompliance in RCTs

In randomized trials, we assign treatment $Z \in \{0, 1\}$, but participants may not comply:
- Some assigned treatment refuse it (D=0 when Z=1)
- Some assigned control seek treatment anyway (D=1 when Z=0)

**Consequence**: Intent-to-Treat (ITT) ≠ Treatment effect on compliers

### Principal Strata

Define strata by potential treatment under each assignment:

| Stratum | D(0) | D(1) | Description |
|---------|------|------|-------------|
| **Compliers** | 0 | 1 | Take treatment iff assigned |
| **Always-takers** | 1 | 1 | Always take treatment |
| **Never-takers** | 0 | 0 | Never take treatment |
| Defiers | 1 | 0 | Do opposite (ruled out by monotonicity) |

### The Key Identity: CACE = LATE

Under IV assumptions (independence, exclusion, monotonicity, relevance):

$$\text{CACE} = \frac{E[Y|Z=1] - E[Y|Z=0]}{E[D|Z=1] - E[D|Z=0]} = \frac{\text{Reduced Form}}{\text{First Stage}}$$

### Visualizing Principal Strata

In [ ]:
# Typical strata proportions in an RCT
strata = ['Compliers', 'Always-takers', 'Never-takers']
proportions = [0.60, 0.20, 0.20]
colors = ['#2ecc71', '#e74c3c', '#3498db']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(strata, proportions, color=colors, edgecolor='black')
ax.set_ylabel('Proportion', fontsize=12)
ax.set_title('Principal Strata in an RCT with Noncompliance', fontsize=14)
ax.set_ylim(0, 0.8)

# Add value labels
for bar, p in zip(bars, proportions):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{p:.0%}', ha='center', fontsize=12, fontweight='bold')

ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Majority threshold')
plt.tight_layout()
plt.show()

print("Compliers are the only stratum where treatment effect is identified!")
print("CACE = treatment effect for compliers only.")

---

## Part 2: CACE Estimation

### Data Generating Process

In [ ]:
def generate_noncompliance_dgp(
    n: int = 1000,
    pi_c: float = 0.60,      # Complier proportion
    pi_a: float = 0.20,      # Always-taker proportion
    pi_n: float = 0.20,      # Never-taker proportion
    cace: float = 2.0,       # True CACE (complier effect)
    mu_a: float = 1.0,       # Always-taker baseline outcome
    mu_n: float = 0.0,       # Never-taker baseline outcome
    mu_c0: float = 0.5,      # Complier outcome when D=0
    seed: int = 42,
) -> dict:
    """Generate RCT data with one-sided noncompliance.
    
    Model:
        Z ~ Bernoulli(0.5)           # Random assignment
        S ~ Multinomial(pi_c, pi_a, pi_n)  # Stratum membership
        D = f(Z, S)                  # Treatment follows stratum rules
        Y = mu_S + CACE * D * I(S=complier) + noise
    
    Returns
    -------
    dict with: outcome, treatment, instrument, stratum, true_cace, proportions
    """
    rng = np.random.default_rng(seed)
    
    # Random assignment
    Z = rng.binomial(1, 0.5, n)
    
    # Draw stratum membership (latent - never observed)
    probs = [pi_c, pi_a, pi_n]
    stratum = rng.choice(['complier', 'always_taker', 'never_taker'], n, p=probs)
    
    # Actual treatment follows stratum rules
    D = np.zeros(n, dtype=int)
    D[stratum == 'always_taker'] = 1              # Always take
    D[stratum == 'never_taker'] = 0               # Never take
    D[(stratum == 'complier') & (Z == 1)] = 1     # Compliers follow Z
    D[(stratum == 'complier') & (Z == 0)] = 0
    
    # Outcomes
    noise = rng.normal(0, 1, n)
    Y = np.zeros(n)
    Y[stratum == 'always_taker'] = mu_a + noise[stratum == 'always_taker']
    Y[stratum == 'never_taker'] = mu_n + noise[stratum == 'never_taker']
    Y[stratum == 'complier'] = mu_c0 + cace * D[stratum == 'complier'] + noise[stratum == 'complier']
    
    return {
        'outcome': Y,
        'treatment': D,
        'instrument': Z,
        'stratum': stratum,  # Latent - for validation only
        'true_cace': cace,
        'true_proportions': {'compliers': pi_c, 'always_takers': pi_a, 'never_takers': pi_n},
    }

In [ ]:
# Generate data
data = generate_noncompliance_dgp(n=1000, cace=2.0, seed=42)

Y = data['outcome']
D = data['treatment']
Z = data['instrument']

print("=== Data Summary ===")
print(f"N: {len(Y)}")
print(f"Assignment rate (Z=1): {Z.mean():.1%}")
print(f"Treatment rate (D=1): {D.mean():.1%}")
print(f"True CACE: {data['true_cace']}")
print(f"\nTrue strata proportions:")
for stratum, prop in data['true_proportions'].items():
    print(f"  {stratum}: {prop:.0%}")

### Run CACE Estimation via 2SLS

In [ ]:
# Estimate CACE
result = cace_2sls(Y, D, Z)

print("=== CACE Estimation Results ===")
print(f"CACE: {result['cace']:.3f}")
print(f"SE: {result['se']:.3f}")
print(f"95% CI: [{result['ci_lower']:.3f}, {result['ci_upper']:.3f}]")
print(f"Z-stat: {result['z_stat']:.2f}, p-value: {result['pvalue']:.4f}")
print(f"\nTrue CACE: {data['true_cace']}")
print(f"Bias: {result['cace'] - data['true_cace']:.3f}")
print(f"Covers true value: {result['ci_lower'] < data['true_cace'] < result['ci_upper']}")

### Strata Proportions and First-Stage Diagnostics

In [ ]:
print("=== Estimated Strata Proportions ===")
sp = result['strata_proportions']
print(f"Compliers: {sp['compliers']:.1%} (true: {data['true_proportions']['compliers']:.0%})")
print(f"Always-takers: {sp['always_takers']:.1%} (true: {data['true_proportions']['always_takers']:.0%})")
print(f"Never-takers: {sp['never_takers']:.1%} (true: {data['true_proportions']['never_takers']:.0%})")

print(f"\n=== First-Stage Diagnostics ===")
print(f"First-stage coefficient: {result['first_stage_coef']:.3f}")
print(f"First-stage F-statistic: {result['first_stage_f']:.1f}")
print(f"Reduced form: {result['reduced_form']:.3f}")

if result['first_stage_f'] < 10:
    print("\n⚠️  WARNING: Weak instrument (F < 10)!")
else:
    print("\n✓ Instrument is strong (F ≥ 10)")

### Compare with Wald Estimator

In [ ]:
# Wald estimator (simple ratio, no covariates)
result_wald = wald_estimator(Y, D, Z)

print("=== Wald Estimator ===")
print(f"CACE: {result_wald['cace']:.3f}")
print(f"SE: {result_wald['se']:.3f}")

print("\n=== Comparison ===")
print(f"2SLS CACE: {result['cace']:.4f}")
print(f"Wald CACE: {result_wald['cace']:.4f}")
print(f"Difference: {abs(result['cace'] - result_wald['cace']):.6f}")
print("\n→ Without covariates, 2SLS and Wald are equivalent!")

---

## Part 3: Bounds (When Assumptions Fail)

When IV assumptions are uncertain, we can compute **bounds** on CACE:
- **Monotonicity bounds**: Assume no defiers, but allow direct effect of Z on Y
- **No-assumption bounds**: Manski worst-case bounds

In [ ]:
# Bounds with monotonicity only
bounds_mono = ps_bounds_monotonicity(Y, D, Z)

# Manski worst-case bounds
bounds_worst = ps_bounds_no_assumption(Y, D, Z)

print("=== Bounds Comparison ===")
print(f"\nWith monotonicity (no exclusion):")
print(f"  CACE ∈ [{bounds_mono['lower_bound']:.2f}, {bounds_mono['upper_bound']:.2f}]")
print(f"  Width: {bounds_mono['bound_width']:.2f}")

print(f"\nNo assumptions (Manski):")
print(f"  CACE ∈ [{bounds_worst['lower_bound']:.2f}, {bounds_worst['upper_bound']:.2f}]")
print(f"  Width: {bounds_worst['bound_width']:.2f}")

print(f"\nPoint estimate (2SLS): {result['cace']:.2f}")
print(f"True CACE: {data['true_cace']}")

In [ ]:
# Visualize bounds vs point estimate
fig, ax = plt.subplots(figsize=(10, 5))

methods = ['Manski\n(No Assumptions)', 'Monotonicity\n(No Exclusion)', '2SLS\n(All Assumptions)']
y_positions = [2, 1, 0]

# Manski bounds
ax.barh(2, bounds_worst['upper_bound'] - bounds_worst['lower_bound'], 
        left=bounds_worst['lower_bound'], height=0.4, color='#e74c3c', alpha=0.6, label='Bounds')

# Monotonicity bounds
ax.barh(1, bounds_mono['upper_bound'] - bounds_mono['lower_bound'],
        left=bounds_mono['lower_bound'], height=0.4, color='#f39c12', alpha=0.6)

# 2SLS CI
ax.barh(0, result['ci_upper'] - result['ci_lower'],
        left=result['ci_lower'], height=0.4, color='#2ecc71', alpha=0.6)
ax.scatter([result['cace']], [0], color='black', s=100, zorder=5, label='Point Estimate')

# True value
ax.axvline(x=data['true_cace'], color='blue', linestyle='--', linewidth=2, label=f"True CACE = {data['true_cace']}")

ax.set_yticks(y_positions)
ax.set_yticklabels(methods)
ax.set_xlabel('CACE', fontsize=12)
ax.set_title('Bounds vs Point Estimates: Assumption Sensitivity', fontsize=14)
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

print("→ Stronger assumptions yield tighter identification.")

---

## Part 4: Dynamic Treatment Regimes (DTR) Concepts

### The Problem: Sequential Treatment Decisions

Many clinical decisions are **sequential**:
- Stage 1: First-line therapy (Drug A vs Drug B)
- Stage 2: If response, continue; if not, switch or augment

**Goal**: Find optimal **treatment regime** $d^*(H) \rightarrow \{0, 1\}$ that maximizes expected outcome.

### Key Insight: Backward Induction

1. Start at **final stage K**: Estimate optimal $d^*_K$ given history
2. Move **backward**: Estimate $d^*_{K-1}, \ldots, d^*_1$
3. Each stage uses **pseudo-outcomes** incorporating future optimal decisions

### Blip Function

The **blip** $\gamma(H)$ is the treatment effect given history:
$$\gamma(H) = E[Y | H, A=1] - E[Y | H, A=0]$$

Optimal regime: $d^*(H) = \mathbb{1}(\gamma(H) > 0)$ (treat if beneficial)

---

## Part 5: Q-Learning

### Data Generating Process

In [ ]:
def generate_dtr_dgp(
    n: int = 500,
    n_covariates: int = 3,
    true_blip: tuple = (1.0, 2.0),  # (ψ₀, ψ₁): blip = ψ₀ + ψ₁*X₀
    seed: int = 42,
) -> dict:
    """Generate single-stage DTR data with known optimal regime.
    
    Model:
        X ~ N(0, 1)^p
        A ~ Bernoulli(0.5)  # Randomized treatment
        Y = X₀ + A*(ψ₀ + ψ₁*X₀) + ε
        
    Optimal regime: d*(X) = I(ψ₀ + ψ₁*X₀ > 0)
    
    Returns
    -------
    dict with: outcome, treatment, covariates, true_blip, true_optimal
    """
    rng = np.random.default_rng(seed)
    
    psi_0, psi_1 = true_blip
    
    # Covariates
    X = rng.normal(0, 1, (n, n_covariates))
    
    # Randomized treatment
    A = rng.binomial(1, 0.5, n)
    
    # Blip function: treatment effect varies with X₀
    blip = psi_0 + psi_1 * X[:, 0]
    
    # Outcome
    noise = rng.normal(0, 1, n)
    Y = X[:, 0] + A * blip + noise
    
    # True optimal regime
    true_optimal = (blip > 0).astype(int)
    
    return {
        'outcome': Y,
        'treatment': A,
        'covariates': X,
        'true_blip': blip,
        'true_blip_coef': true_blip,
        'true_optimal': true_optimal,
    }

In [ ]:
# Generate DTR data
dtr_data = generate_dtr_dgp(n=500, true_blip=(1.0, 2.0), seed=42)

Y_dtr = dtr_data['outcome']
A_dtr = dtr_data['treatment']
X_dtr = dtr_data['covariates']

print("=== DTR Data Summary ===")
print(f"N: {len(Y_dtr)}")
print(f"Treatment rate: {A_dtr.mean():.1%}")
print(f"Covariates: {X_dtr.shape[1]}")
print(f"True blip coefficients: ψ₀={dtr_data['true_blip_coef'][0]}, ψ₁={dtr_data['true_blip_coef'][1]}")
print(f"True optimal treatment rate: {dtr_data['true_optimal'].mean():.1%}")

### Run Q-Learning

In [ ]:
# Q-learning estimation
result_q = q_learning_single_stage(Y_dtr, A_dtr, X_dtr)

print("=== Q-Learning Results ===")
print(f"Optimal value: {result_q.value_estimate:.3f} (SE: {result_q.value_se:.3f})")
print(f"95% CI: [{result_q.value_ci_lower:.3f}, {result_q.value_ci_upper:.3f}]")

print(f"\n=== Blip Coefficients ===")
blip_coef = result_q.blip_coefficients[0]  # Stage 1 (only stage)
blip_se = result_q.blip_se[0]
print(f"Estimated ψ₀: {blip_coef[0]:.3f} (SE: {blip_se[0]:.3f}) [True: {dtr_data['true_blip_coef'][0]}]")
print(f"Estimated ψ₁: {blip_coef[1]:.3f} (SE: {blip_se[1]:.3f}) [True: {dtr_data['true_blip_coef'][1]}]")

### Optimal Regime Accuracy

In [ ]:
# Predict optimal treatment
predicted_optimal = result_q.predict_optimal_treatment(X_dtr, stage=1)

# Compare to true optimal
accuracy = np.mean(predicted_optimal == dtr_data['true_optimal'])

print(f"=== Regime Accuracy ===")
print(f"Predicted optimal treatment rate: {predicted_optimal.mean():.1%}")
print(f"True optimal treatment rate: {dtr_data['true_optimal'].mean():.1%}")
print(f"Accuracy: {accuracy:.1%}")

if accuracy >= 0.90:
    print("\n✓ Excellent regime recovery (≥90%)")
else:
    print(f"\n⚠️  Regime accuracy below target: {accuracy:.1%} < 90%")

In [ ]:
# Visualize decision boundary
fig, ax = plt.subplots(figsize=(10, 6))

# Scatter plot: X₀ vs predicted treatment
treat_mask = predicted_optimal == 1
ax.scatter(X_dtr[treat_mask, 0], Y_dtr[treat_mask], alpha=0.5, label='Predicted: Treat', c='#2ecc71')
ax.scatter(X_dtr[~treat_mask, 0], Y_dtr[~treat_mask], alpha=0.5, label='Predicted: Control', c='#e74c3c')

# True decision boundary: ψ₀ + ψ₁*X₀ = 0 → X₀ = -ψ₀/ψ₁
psi_0, psi_1 = dtr_data['true_blip_coef']
true_boundary = -psi_0 / psi_1
ax.axvline(x=true_boundary, color='blue', linestyle='--', linewidth=2, label=f'True Boundary: X₀ = {true_boundary:.2f}')

# Estimated boundary
if abs(blip_coef[1]) > 1e-6:
    est_boundary = -blip_coef[0] / blip_coef[1]
    ax.axvline(x=est_boundary, color='orange', linestyle=':', linewidth=2, label=f'Estimated: X₀ = {est_boundary:.2f}')

ax.set_xlabel('X₀ (First Covariate)', fontsize=12)
ax.set_ylabel('Outcome Y', fontsize=12)
ax.set_title('Q-Learning: Optimal Treatment Regime', fontsize=14)
ax.legend()
plt.tight_layout()
plt.show()

print(f"→ Treat when X₀ > {true_boundary:.2f} (blip becomes positive)")

---

## Part 6: A-Learning (Doubly Robust)

**A-learning** differs from Q-learning:
- Q-learning: Model full Q-function $Q(H, A) = E[Y | H, A]$
- A-learning: Model **contrast** (blip) $\gamma(H) = Q(H, 1) - Q(H, 0)$ directly

**Double Robustness**: A-learning is consistent if EITHER:
1. Propensity model $P(A=1|H)$ is correct, OR
2. Baseline outcome model $E[Y|H, A=0]$ is correct

In [ ]:
# A-learning estimation
result_a = a_learning_single_stage(Y_dtr, A_dtr, X_dtr)

print("=== A-Learning Results ===")
print(f"Optimal value: {result_a.value_estimate:.3f} (SE: {result_a.value_se:.3f})")
print(f"95% CI: [{result_a.value_ci_lower:.3f}, {result_a.value_ci_upper:.3f}]")
print(f"Doubly robust: {result_a.doubly_robust}")

print(f"\n=== Blip Coefficients ===")
blip_coef_a = result_a.blip_coefficients[0]
blip_se_a = result_a.blip_se[0]
print(f"Estimated ψ₀: {blip_coef_a[0]:.3f} (SE: {blip_se_a[0]:.3f}) [True: {dtr_data['true_blip_coef'][0]}]")
print(f"Estimated ψ₁: {blip_coef_a[1]:.3f} (SE: {blip_se_a[1]:.3f}) [True: {dtr_data['true_blip_coef'][1]}]")

In [ ]:
# Compare Q-learning vs A-learning
print("=== Q-Learning vs A-Learning Comparison ===")
print(f"\n{'Metric':<25} {'Q-Learning':>15} {'A-Learning':>15}")
print("-" * 60)
print(f"{'Optimal value':<25} {result_q.value_estimate:>15.3f} {result_a.value_estimate:>15.3f}")
print(f"{'Value SE':<25} {result_q.value_se:>15.3f} {result_a.value_se:>15.3f}")
print(f"{'ψ₀ estimate':<25} {result_q.blip_coefficients[0][0]:>15.3f} {result_a.blip_coefficients[0][0]:>15.3f}")
print(f"{'ψ₁ estimate':<25} {result_q.blip_coefficients[0][1]:>15.3f} {result_a.blip_coefficients[0][1]:>15.3f}")

# Regime agreement
predicted_a = result_a.predict_optimal_treatment(X_dtr, stage=1)
agreement = np.mean(predicted_optimal == predicted_a)
print(f"\nRegime agreement: {agreement:.1%}")

accuracy_a = np.mean(predicted_a == dtr_data['true_optimal'])
print(f"\nQ-learning accuracy: {accuracy:.1%}")
print(f"A-learning accuracy: {accuracy_a:.1%}")

---

## Part 7: When to Use What?

| Scenario | Recommended Method | Key Consideration |
|----------|-------------------|-------------------|
| RCT with noncompliance | **CACE (2SLS)** | Need valid instrument |
| Uncertain IV assumptions | **Bounds** | Trade precision for robustness |
| Sequential treatment | **Q-learning** | Need correct Q-model |
| Model uncertainty | **A-learning** | Double robustness |

### Decision Tree

```
Is treatment sequential?
├── NO: Single-stage
│   ├── Have instrument (Z)?
│   │   ├── YES: Is instrument strong (F>10)?
│   │   │   ├── YES → CACE (2SLS)
│   │   │   └── NO → Bounds or robust inference
│   │   └── NO: No principal stratification
│   └── Concerned about assumptions? → Bounds
│
└── YES: Multi-stage
    ├── Propensity known/modeled well? → A-learning
    └── Outcome model reliable? → Q-learning
```

### Practical Considerations

In [ ]:
# Effect of sample size on CACE precision
sample_sizes = [100, 250, 500, 1000, 2000]
ses = []

for n in sample_sizes:
    data_n = generate_noncompliance_dgp(n=n, cace=2.0, seed=42)
    result_n = cace_2sls(data_n['outcome'], data_n['treatment'], data_n['instrument'])
    ses.append(result_n['se'])

print("=== CACE Precision vs Sample Size ===")
print(f"{'N':>10} {'SE':>10} {'CI Width':>12}")
print("-" * 35)
for n, se in zip(sample_sizes, ses):
    print(f"{n:>10} {se:>10.3f} {1.96*2*se:>12.3f}")

print("\n→ SE ∝ 1/√N (standard rate)")

In [ ]:
# Weak instrument effect
print("=== Weak Instrument Effect ===")

# Strong instrument (60% compliers)
data_strong = generate_noncompliance_dgp(n=500, pi_c=0.60, pi_a=0.20, pi_n=0.20, seed=42)
result_strong = cace_2sls(data_strong['outcome'], data_strong['treatment'], data_strong['instrument'])

# Weak instrument (10% compliers)
data_weak = generate_noncompliance_dgp(n=500, pi_c=0.10, pi_a=0.45, pi_n=0.45, seed=42)
result_weak = cace_2sls(data_weak['outcome'], data_weak['treatment'], data_weak['instrument'])

print(f"{'Instrument':>15} {'Compliers':>12} {'F-stat':>10} {'SE':>10} {'CI Width':>12}")
print("-" * 60)
print(f"{'Strong':>15} {data_strong['true_proportions']['compliers']:>12.0%} {result_strong['first_stage_f']:>10.1f} {result_strong['se']:>10.3f} {1.96*2*result_strong['se']:>12.3f}")
print(f"{'Weak':>15} {data_weak['true_proportions']['compliers']:>12.0%} {result_weak['first_stage_f']:>10.1f} {result_weak['se']:>10.3f} {1.96*2*result_weak['se']:>12.3f}")

print("\n→ Weak instruments dramatically inflate SE!")

---

## Summary

### Key Takeaways

1. **Principal Stratification** handles noncompliance by identifying effects for **compliers only**
2. **CACE = LATE** under IV assumptions (independence, exclusion, monotonicity, relevance)
3. **Bounds** trade precision for robustness when assumptions are uncertain
4. **Q-learning** uses backward induction to find optimal treatment regimes
5. **A-learning** provides **double robustness** - consistent if either model correct

### References

1. **Angrist, Imbens, Rubin (1996)** - "Identification of Causal Effects Using Instrumental Variables"
2. **Murphy (2003)** - "Optimal Dynamic Treatment Regimes"
3. **Robins (2004)** - "Optimal Structural Nested Models for Optimal Sequential Decisions"
4. **Schulte et al. (2014)** - "Q- and A-Learning Methods for Estimating Optimal Dynamic Treatment Regimes"

---

*Created: Session 127 - Causal Inference Mastery*